# SPY 0DTE Iron Condor — Quick Runner

Lean Colab notebook. Cell 1 only does `git clone` (no pip install) — should finish in <30 s on any runtime.

Run cells **top to bottom** with the ▶ button. No editing needed.

## 1. Setup (just clone the repo)

In [ ]:
import os, sys
print('a) preparing...')
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
if not os.path.isdir(REPO):
    print('b) cloning repo (a few seconds)...')
    !git clone --quiet -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO}
else:
    print('b) repo exists; pulling latest...')
    !cd {REPO} && git pull --quiet
print('c) adding src/ to PYTHONPATH...')
SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Run the sweep (240 configs)
30 days, `fixed_1.0x3` strike, RSI 14, 4 RSI thresholds, all PT/SL/cutoffs.

**First time on a fresh runtime: 10–15 min** (Polygon cache is cold). Reruns are ~3 min.

In [ ]:
!python -m iron_condor.cli --sweep

## 4. Top configs

In [ ]:
import pandas as pd
summary = pd.read_csv('results/sweep_summary.csv')
summary.head(20)

## 5. Timing analysis — winners vs losers
Look at `profit p75/p90/max` for winning trades — that's where the early time-stop should sit.

In [ ]:
from iron_condor.metrics import analyze_timing
trades = pd.read_csv('results/sweep_trades.csv')
print('=== All configs combined ===')
print(analyze_timing(trades))

In [ ]:
# Top-5 configs broken out
for cfg in summary.head(5)['config'].tolist():
    print(f'\n--- {cfg} ---')
    print(analyze_timing(trades[trades['config'] == cfg]))

In [ ]:
# By RSI threshold
trades['rsi_thresh'] = trades['config'].str.extract(r'rsi\d+_(\d+-\d+)')
for thr in trades['rsi_thresh'].dropna().unique():
    print(f'\n--- RSI threshold {thr} ---')
    print(analyze_timing(trades[trades['rsi_thresh'] == thr]))